# Exchange Rate Forecasting

## Project Overview

Forecasts a **daily exchange rate** (e.g., EUR/USD) over a 14-day horizon. Synthetic data spans 2 years with mean-reverting behavior, central bank intervention effects, and mild seasonal patterns.

**Models**: Naive, LazyPredict, FLAML, StatsForecast. **Horizon**: 14 days.

## Learning Objectives

1. Understand time-series patterns (trend, seasonality, noise)
2. Build naive and seasonal-naive baselines
3. Engineer lag and rolling features for tabular ML
4. Use LazyPredict for quick regression benchmarking
5. Apply FLAML AutoML (excluding XGBoost due to compatibility)
6. Use StatsForecast (AutoARIMA, AutoETS, SeasonalNaive)
7. Compare all approaches with MAE / RMSE / MAPE

## Problem Statement

Given historical daily exchange rates, predict the next 14 days. FX forecasting supports international trade, hedging, and central bank policy analysis.

## Why This Project Matters

The foreign exchange market is the largest financial market ($7.5 trillion daily volume). Exchange rate movements affect exporters, importers, multinational corporations, and central banks. Even small improvements in FX forecasting translate to significant economic value in hedging and trade finance.

## Dataset Overview

Synthetic dataset:
- 730 days (2 years) of daily exchange rate
- Mean-reverting around purchasing power parity
- Central bank intervention effects
- Interest rate differential drift
- Mild end-of-month / quarter effects

| Column | Description |
|--------|-------------|
| `date` | Date |
| `rate` | Daily exchange rate (e.g., EUR/USD) |

## Dataset Source and License Notes

Synthetically generated for educational purposes. No external download required.

## Environment Setup

In [1]:
import subprocess, importlib
for pkg in ['numpy', 'pandas', 'matplotlib', 'scikit-learn', 'statsforecast', 'flaml', 'lazypredict']:
    try:
        importlib.import_module(pkg.replace('-', '_').split('[')[0])
    except ImportError:
        subprocess.check_call(['pip', 'install', '-q', pkg])
print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## Imports

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import mean_absolute_error, mean_squared_error
print("Imports complete.")

Imports complete.


## Configuration / Constants

In [3]:
SEED = 42
N_POINTS = 730
HORIZON = 14
VAL_SIZE = 14
TEST_SIZE = 14
SEASON_LENGTH = 7
FREQ = 'D'
TARGET = 'rate'
np.random.seed(SEED)
print(f"Config: {N_POINTS} points, horizon={HORIZON}, season={SEASON_LENGTH}")

Config: 730 points, horizon=14, season=7


## Dataset Generation

In [4]:
dates = pd.date_range(start='2022-01-01', periods=N_POINTS, freq='D')
t = np.arange(N_POINTS)

# Mean-reverting around 1.10 with slow drift
equilibrium = 1.10 + 0.00005 * t
rate = np.zeros(N_POINTS)
rate[0] = 1.10

theta = 0.02  # mean reversion speed
sigma = 0.004  # daily volatility

for i in range(1, N_POINTS):
    rate[i] = rate[i-1] + theta * (equilibrium[i] - rate[i-1]) + sigma * np.random.normal()

# Central bank intervention effects (rare, large)
for s in range(0, N_POINTS, 250):
    intervention_day = min(s + np.random.randint(100, 200), N_POINTS - 3)
    shift = np.random.choice([-0.02, 0.02])
    for j in range(min(3, N_POINTS - intervention_day)):
        rate[intervention_day + j] += shift * np.exp(-0.5 * j)

# Mild month-end effects
for i in range(N_POINTS):
    if dates[i].day >= 28:
        rate[i] += np.random.normal(0, 0.001)

rate = np.maximum(rate, 0.5).round(4)

df = pd.DataFrame({'date': dates, 'rate': rate})
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (730, 2)


,date,rate
0,2022-01-01,1.1000
1,2022-01-02,1.1020
2,2022-01-03,1.1014
3,2022-01-04,1.1040
4,2022-01-05,1.1100


## Data Validation Checks

In [5]:
assert df.isnull().sum().sum() == 0, "Missing values"
assert (df[TARGET] > 0).all(), "Non-positive values"
assert df['date'].is_monotonic_increasing, "Not sorted"
assert len(df) == N_POINTS, "Row count mismatch"
print("All validation checks passed.")

All validation checks passed.


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
axes[0, 0].plot(df['date'], df[TARGET], linewidth=0.6)
axes[0, 0].set_title('rate Over Time')
axes[0, 1].hist(df[TARGET], bins=30, edgecolor='black', alpha=0.7)
axes[0, 1].set_title('Distribution')
from pandas.plotting import autocorrelation_plot
autocorrelation_plot(df[TARGET], ax=axes[1, 0])
axes[1, 0].set_xlim(0, min(60, N_POINTS // 2))
axes[1, 0].set_title('Autocorrelation')
rw = min(SEASON_LENGTH * 2, N_POINTS // 4)
axes[1, 1].plot(df['date'], df[TARGET].rolling(rw).mean())
axes[1, 1].set_title(f'Rolling {rw}-period Mean')
plt.tight_layout()
plt.savefig('eda.png', dpi=100, bbox_inches='tight')
plt.show()
print("EDA complete.")

EDA complete.


## Target Analysis

In [7]:
print("rate Statistics:")
print(df[TARGET].describe())
print(f"\nCV: {df[TARGET].std() / df[TARGET].mean():.3f}")
print(f"Skewness: {df[TARGET].skew():.3f}")

rate Statistics:
count    730.000000
mean       1.114366
std        0.018956
min        1.065900
25%        1.101400
50%        1.116400
75%        1.128075
max        1.157500
Name: rate, dtype: float64

CV: 0.017
Skewness: -0.322


## Train / Validation / Test Split Strategy

Time-aware split (no shuffling):
- **Train**: all except last 28
- **Validation**: 14 points
- **Test**: last 14 points

In [8]:
train = df.iloc[:-(VAL_SIZE + TEST_SIZE)].copy()
val = df.iloc[-(VAL_SIZE + TEST_SIZE):-TEST_SIZE].copy()
test = df.iloc[-TEST_SIZE:].copy()
print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

Train: 702, Val: 14, Test: 14


## Preprocessing Strategy

Minimal preprocessing — tree models handle raw features. Features created next.

In [9]:
df_full = df.copy().sort_values('date').reset_index(drop=True)
print('Data ready.')

Data ready.


## Feature Engineering

In [10]:
def create_features(data):
    d = data.copy()
    d['dow'] = d['date'].dt.dayofweek
    d['month'] = d['date'].dt.month
    d['day'] = d['date'].dt.day
    d['week_of_year'] = d['date'].dt.isocalendar().week.astype(int)
    for lag in [1, 7, 14, 28]:
        d[f'lag_{lag}'] = d[TARGET].shift(lag)
    for w in [7, 14, 28]:
        d[f'rmean_{w}'] = d[TARGET].shift(1).rolling(w).mean()
        d[f'rstd_{w}'] = d[TARGET].shift(1).rolling(w).std()
    return d

df_feat = create_features(df_full).dropna().reset_index(drop=True)
feature_cols = [c for c in df_feat.columns if c not in ['date', TARGET]]
print(f"Features: {len(feature_cols)} columns, {len(df_feat)} rows")

Features: 14 columns, 702 rows


## Baseline Model — Naive Forecasts

In [11]:
def calc_metrics(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.maximum(y_true, 1))) * 100
    print(f"{name:30s} | MAE: {mae:10.1f} | RMSE: {rmse:10.1f} | MAPE: {mape:5.2f}%")
    return {'model': name, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

results = []
naive_pred = np.full(TEST_SIZE, train[TARGET].iloc[-1])
results.append(calc_metrics(test[TARGET].values, naive_pred, "Naive (Last Value)"))

src = df_full[TARGET].values
ti = len(df_full) - TEST_SIZE
sn_pred = src[ti - SEASON_LENGTH:ti - SEASON_LENGTH + TEST_SIZE]
results.append(calc_metrics(test[TARGET].values, sn_pred, f"Seasonal Naive ({SEASON_LENGTH})"))

Naive (Last Value)             | MAE:        0.0 | RMSE:        0.0 | MAPE:  0.35%
Seasonal Naive (7)             | MAE:        0.0 | RMSE:        0.0 | MAPE:  0.56%


## LazyPredict Benchmark (Lag-Feature Tabular)

In [12]:
from lazypredict.Supervised import LazyRegressor

ct = len(df_feat) - TEST_SIZE
cv = ct - VAL_SIZE
X_tr = df_feat.iloc[:cv][feature_cols]
y_tr = df_feat.iloc[:cv][TARGET]
X_va = df_feat.iloc[cv:ct][feature_cols]
y_va = df_feat.iloc[cv:ct][TARGET]

reg = LazyRegressor(verbose=0, ignore_warnings=True, custom_metric=None, predictions=True)
lp_models, lp_preds = reg.fit(X_tr, X_va, y_tr, y_va)
print("\nLazyPredict Top 10:")
print(lp_models.head(10).to_string())


LazyPredict Top 10:
                            Adjusted R-Squared     R-Squared      RMSE  Time Taken
Model                                                                             
KernelRidge                      838148.800994 -64471.907769  1.113777    0.025405
GaussianProcessRegressor         103538.473327  -7963.421025  0.391459    0.049509
MLPRegressor                       4730.627930   -362.817533  0.083666    0.210456
PassiveAggressiveRegressor          592.751055    -44.519312  0.029594    0.010882
SVR                                  86.078374     -5.544490  0.011221    0.014199
ExtraTreeRegressor                   67.643150     -4.126396  0.009932    0.011293
LassoLars                            51.186846     -2.860527  0.008619    0.016492
ElasticNet                           51.186846     -2.860527  0.008619    0.008472
DummyRegressor                       51.186846     -2.860527  0.008619    0.007332
Lasso                                51.186846     -2.860527  0.00

## FLAML AutoML Run

In [13]:
from flaml import AutoML

automl = AutoML()
automl.fit(
    X_train=X_tr, y_train=y_tr,
    task='regression', time_budget=30, metric='mae',
    estimator_list=['lgbm', 'rf', 'extra_tree', 'catboost'],
    seed=SEED, verbose=0
)
print(f"FLAML best: {automl.best_estimator}")
flaml_val = automl.predict(X_va)
results.append(calc_metrics(y_va.values, flaml_val, f"FLAML ({automl.best_estimator})"))

X_te = df_feat.iloc[ct:][feature_cols]
y_te = df_feat.iloc[ct:][TARGET]
flaml_test = automl.predict(X_te)
results.append(calc_metrics(y_te.values, flaml_test, f"FLAML Test ({automl.best_estimator})"))

FLAML best: rf
FLAML (rf)                     | MAE:        0.0 | RMSE:        0.0 | MAPE:  0.37%
FLAML Test (rf)                | MAE:        0.0 | RMSE:        0.0 | MAPE:  0.25%


## StatsForecast — Statistical Forecasting

In [14]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive as SFSeasonalNaive

sf_df = df_full[['date', TARGET]].copy()
sf_df.columns = ['ds', 'y']
sf_df['unique_id'] = 'series1'
sf_df = sf_df[['unique_id', 'ds', 'y']]

sf_train = sf_df.iloc[:-TEST_SIZE]
sf = StatsForecast(
    models=[AutoARIMA(season_length=SEASON_LENGTH), AutoETS(season_length=SEASON_LENGTH),
            SFSeasonalNaive(season_length=SEASON_LENGTH)],
    freq=FREQ, n_jobs=1
)
sf.fit(sf_train)
sf_preds = sf.predict(h=TEST_SIZE).reset_index()

y_test_sf = test[TARGET].values
for col in ['AutoARIMA', 'AutoETS', 'SeasonalNaive']:
    if col in sf_preds.columns:
        results.append(calc_metrics(y_test_sf, sf_preds[col].values, f"SF {col}"))
print("StatsForecast complete.")

SF AutoARIMA                   | MAE:        0.0 | RMSE:        0.0 | MAPE:  0.30%
SF AutoETS                     | MAE:        0.0 | RMSE:        0.0 | MAPE:  0.30%
SF SeasonalNaive               | MAE:        0.0 | RMSE:        0.0 | MAPE:  0.35%
StatsForecast complete.


## Top 3 Model Selection

In [15]:
results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
print("All Results:")
print(results_df.to_string(index=False))
print("\nTop 3:")
print(results_df.head(3).to_string(index=False))

All Results:
             model      MAE     RMSE     MAPE
   FLAML Test (rf) 0.002765 0.003474 0.246305
        SF AutoETS 0.003336 0.003975 0.296387
      SF AutoARIMA 0.003396 0.003980 0.301838
Naive (Last Value) 0.003879 0.004834 0.345849
  SF SeasonalNaive 0.003907 0.005150 0.346866
        FLAML (rf) 0.004188 0.004950 0.373490
Seasonal Naive (7) 0.006286 0.006913 0.559090

Top 3:
          model      MAE     RMSE     MAPE
FLAML Test (rf) 0.002765 0.003474 0.246305
     SF AutoETS 0.003336 0.003975 0.296387
   SF AutoARIMA 0.003396 0.003980 0.301838


## Final Training and Evaluation of Top 3

In [16]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test['date'].values, test[TARGET].values, 'ko-', label='Actual', ms=4)
ax.plot(test['date'].values, flaml_test, 's--', label=f'FLAML ({automl.best_estimator})', ms=4)
for col in ['AutoARIMA', 'AutoETS']:
    if col in sf_preds.columns:
        ax.plot(test['date'].values[:len(sf_preds)], sf_preds[col].values, 'o--', label=f'SF {col}', ms=4)
ax.set_title('Forecast vs Actual — Test Set')
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

## Error Analysis

In [17]:
errors = test[TARGET].values - flaml_test
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(len(errors)), errors, alpha=0.7)
axes[0].set_title('Residuals (Actual - FLAML)')
axes[0].axhline(y=0, color='r', linestyle='--')
axes[1].hist(errors, bins=10, edgecolor='black', alpha=0.7)
axes[1].set_title('Residual Distribution')
plt.tight_layout()
plt.savefig('error_analysis.png', dpi=100, bbox_inches='tight')
plt.show()
print(f"Mean residual: {np.mean(errors):.2f}, Std: {np.std(errors):.2f}")

Mean residual: -0.00, Std: 0.00


## Interpretation and Business Insight

- Exchange rates mean-revert around fundamental equilibrium (PPP)
- Central bank interventions create sharp, temporary moves
- Day-to-day changes are small relative to the level
- Interest rate differentials drive slow drift
- FX markets are highly efficient — the random walk is hard to beat

## Limitations

1. Synthetic — real FX depends on interest rates, inflation, trade balances, politics
2. No interest rate differential data
3. No carry trade or flow data
4. Single currency pair — cross-rate relationships ignored
5. No central bank communication or policy meeting data

## How to Improve This Project

1. Add interest rate differentials (key FX driver)
2. Include trade balance and current account data
3. Use carry trade signals as features
4. Add central bank meeting dates and policy expectations
5. Test with PPP and BEER models for fundamental fair value

## Production Considerations

- FX forecasting for corporate hedging programs
- Integration with treasury management systems
- Automated hedge ratio optimization
- Real-time risk exposure monitoring

## Common Mistakes

1. Expecting high accuracy — FX is near random walk
2. Ignoring transaction costs (bid-ask spread)
3. Not accounting for interest rate differentials
4. Using nominal rates without inflation adjustment
5. Overfitting to historical patterns that reflect past policy regimes

## Mini Challenge / Exercises

1. Test whether daily returns are autocorrelated (Ljung-Box test)
2. Estimate the mean-reversion half-life
3. Compare forecast accuracy when rate is near vs far from equilibrium
4. Build a simple carry trade signal and test profitability

## Final Summary / Key Takeaways

1. FX markets are among the most efficient — hard to forecast
2. Mean reversion around fundamentals provides some signal
3. Central bank actions create the largest unpredictable moves
4. Interest rate differentials are the key fundamental driver
5. Hedging optimization is more valuable than directional prediction

In [18]:
import json
metrics = {
    'project': 'Exchange Rate Forecasting',
    'best_model': results_df.iloc[0]['model'],
    'best_MAE': float(results_df.iloc[0]['MAE']),
    'best_RMSE': float(results_df.iloc[0]['RMSE']),
    'best_MAPE': float(results_df.iloc[0]['MAPE']),
    'all_results': results_df.to_dict('records')
}
with open('metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved.")
print("\n=== Exchange Rate Forecasting — Complete ===")

Metrics saved.

=== Exchange Rate Forecasting — Complete ===
